In [28]:
import pandas as pd
import numpy as np
import pymongo
import os
from dotenv import load_dotenv
import datetime as dt

data = pd.read_csv('./database/healthcare_dataset.csv')



In [4]:
def verify_data(data):
    print("Aperçu des données :")
    print(data.head())
    print("\nInformations sur les données :")
    print(data.info())
    print("\nStatistiques descriptives :")
    print(data.describe())

In [ ]:
def clean_data(file_path):
    data = pd.read_csv('./database/healthcare_dataset.csv')
    # Ajout d'un index unique pour chaque patient
    data['patient_id'] = data.index
    # Supprimer les doublons et les valeurs manquantes
    data = data.drop_duplicates()
    data = data.dropna()
    # Renommer les colonnes pour plus de clarté
    data = data.rename(columns={"patient_id":"patient_id","Name":"name","Age":"age","Gender":"gender","Blood Type":"blood_type",
            "Medical Condition":"medical_condition","Date of Admission":"date_of_admission","Doctor":"doctor","Hospital":"hospital","Insurance Provider":"insurance_provider",
            "Billing Amount":"billing_amount","Room Number":"room_number","Admission Type":"admission_type","Discharge Date":"discharge_date","Medication":"medication","Test Results":"test_results"})
    # Normaliser les noms et arrondir les montants
    data['name'] = data['name'].str.title()
    data['name'] = data['name'].str.strip()
    data['billing_amount'] = data['billing_amount'].round(2)
    # Normaliser les dates
    data['date_of_admission'] = pd.to_datetime(data['date_of_admission'], unit='ms')
    data['discharge_date'] = pd.to_datetime(data['discharge_date'], unit='ms')
    return data

In [44]:
cleaned_data = clean_data(data)

In [38]:
ordre =["patient_id",
            "name",
            "age",
            "gender",
            "blood_type",
            "medical_condition",
            "date_of_admission",
            "doctor",
            "hospital",
            "insurance_provider",
            "billing_amount",
            "room_number",
            "admission_type",
            "discharge_date",
            "medication",
            "test_results"]
cleaned_data = cleaned_data[ordre]

In [45]:
cleaned_data.dtypes

name                             str
age                            int64
gender                           str
blood_type                       str
medical_condition                str
date_of_admission     datetime64[us]
doctor                           str
hospital                         str
insurance_provider               str
billing_amount               float64
room_number                    int64
admission_type                   str
discharge_date        datetime64[us]
medication                       str
test_results                     str
patient_id                     int64
dtype: object

In [36]:
cleaned_data.head()

,name,age,gender,blood_type,medical_condition,date_of_admission,doctor,hospital,insurance_provider,billing_amount,room_number,admission_type,discharge_date,medication,test_results,patient_id
0,Bobby Jackson,30,Male,B-,Cancer,2024-01-31,Matthew Smith,Sons and Miller,Blue Cross,18856.28,328,Urgent,2024-02-02,Paracetamol,Normal,0
1,Leslie Terry,62,Male,A+,Obesity,2019-08-20,Samantha Davies,Kim Inc,Medicare,33643.33,265,Emergency,2019-08-26,Ibuprofen,Inconclusive,1
2,Danny Smith,76,Female,A-,Obesity,2022-09-22,Tiffany Mitchell,Cook PLC,Aetna,27955.10,205,Emergency,2022-10-07,Aspirin,Normal,2
3,Andrew Watts,28,Female,O+,Diabetes,2020-11-18,Kevin Wells,"Hernandez Rogers and Vang,",Medicare,37909.78,450,Elective,2020-12-18,Ibuprofen,Abnormal,3
4,Adrienne Bell,43,Female,AB+,Cancer,2022-09-19,Kathleen Hanna,White-White,Aetna,14238.32,458,Urgent,2022-10-09,Penicillin,Abnormal,4


In [5]:
load_dotenv()
file_path = os.environ.get("OUTPUT_DATA_PATH")
db_name = os.environ.get("MONGO_INITDB_DATABASE")
collection_name = os.environ.get("COLLECTION")
user = os.environ.get("MONGO_INITDB_ROOT_USERNAME")
password = os.environ.get("MONGO_INITDB_ROOT_PASSWORD")
mongodb_uri = f"mongodb://0.0.0.0:27017/"

In [6]:
mongodb_uri

'mongodb://0.0.0.0:27017/'

In [7]:

def import_to_mongodb(data,mongodb_uri):
    client = pymongo.MongoClient(mongodb_uri)
    db = client['DataSoluTech']
    chunk_size = 1000
    num_chunks = int((len(cleaned_data))/chunk_size)
    chunks = []
        
    for i in range(num_chunks):
        start = chunk_size * i
        stop = start + chunk_size
        chunks.append(cleaned_data[start:stop])
    # itérer sur les blocs    
    for i in range(num_chunks):
        db.healthcare.insert_many(chunks[i].to_dict('records'))
        print(f"Chunk {i+1}/{num_chunks} imported successfully.")
        
        


In [ ]:
cleaned_data

Index(['Name', 'Age', 'Gender', 'Blood Type', 'Medical Condition',
       'Date of Admission', 'Doctor', 'Hospital', 'Insurance Provider',
       'Billing Amount', 'Room Number', 'Admission Type', 'Discharge Date',
       'Medication', 'Test Results'],
      dtype='str')

In [ ]:
db_schema = {
    'Name': {
        'type': 'string',
        'minlength': 1,
        'required': True,
    },
    'Age': {
        'type': 'int',
        'minlength': 1,
        'required': True,
    },
    'Gender': {
        'type': 'string',
        "required": False,
        'enum': ['Male', 'Female', 'Other']
    },
    'Blood Type': {
        'type': 'string',
        'required': True,
        'enum': ['A+', 'A-', 'B+', 'B-', 'AB+', 'AB-', 'O+', 'O-']
    },
    'Medical Condition': {
        'type': 'string',
        'required': True,
    },
    'Date of Admission': {
        'type': 'date',
        'required': True,
    },
    'Doctor': {
        'type': 'string',
        'required': True,
    },
    "Hospital": {
        "type": "string",
        "required": True
    },
    'Insurance Provider': {
        'type': 'string',
        'required': True,
    },
    'Billing Amount': {
        'type': 'float',
        'required': True,
    },
    'Billing Amount': {
        'type': 'float',
        'required': True,
    },
    'Billing Amount': {
        'type': 'float',
        'required': True,
    },
    'Room Number': {
        'type': 'int',
        'required': True,
    },
    'Admission Type': {
        'type': 'string',
        'required': True,
        'enum': ['Emergency', 'Elective', 'Urgent']
    },
    'Discharge Date': {
        'type': 'date',
        'required': True,
    },
    'Medication': {
        'type': 'string',
        'required': True,
    },
     'Test Results': {
        'type': 'string',
        'required': True,
        'enum' : ['Normal', 'Abnormal', 'Inconclusive']
    },
}
                
